# Promote, reject, and roll back model versions

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


In [ ]:
!pip install mlflow

## Optional MLflow setup
The first cell installs MLflow. The notebook uses a temporary local tracking store by default; start a server only when testing a shared tracking endpoint.

```bash
mlflow server --host 127.0.0.1 --port 5000
```

Set the tracking URI through your environment when needed. Stop the local server with Ctrl-C; the local manifest remains the offline fallback.


**Set up a deterministic source run**


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
from fraudtwin.ml import heuristic_predictions

dataset = data.require_dataset()
policy_rows = dataset.rows[:20]
scores = heuristic_predictions(policy_rows)
metrics = {"rows": len(scores), "mean_score": sum(s.fraud_score for s in scores) / len(scores)}
print(metrics)

**Run the core operation**


In [ ]:
candidate = {"version": "candidate-1", "schema": "pit-v1", "metrics": metrics}
print(candidate)

**Measure and interpret the result**


In [ ]:
registry = {"Production": None, "Staging": candidate}
registry["Production"] = registry["Staging"]
print("promoted:", registry["Production"]["version"])

**Exercise a parameter or failure mode**


In [ ]:
registry["Staging"] = {**candidate, "version": "candidate-2"}
registry["Production"] = candidate
print("rollback:", registry["Production"]["version"])

**Write a compact artifact and fingerprint**


In [ ]:
assert registry["Production"]["schema"] == "pit-v1"
print("Optional MLflow registry calls can replace this local state machine.")

**Verify invariants and clean up**


In [ ]:
# A compact inspection is more useful than printing an entire run.
print(
    payments.select(
        [
            c
            for c in ("payment_id", "amount", "initiated_at", "payer_account_id")
            if c in payments.columns
        ]
    ).head(8)
)
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})

**Optional service integration**


In [ ]:
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

**Review the expected outcome**


In [ ]:
try:
    import mlflow
    from mlflow.tracking import MlflowClient

    with TemporaryDirectory(prefix="fraudtwin-mlflow-") as mlflow_dir:
        mlflow.set_tracking_uri(mlflow_dir)
        mlflow.set_experiment("fraudtwin-tutorial")
        with mlflow.start_run() as run:
            mlflow.log_params({"tutorial_id": 18, "schema": candidate["schema"]})
            mlflow.log_metric("mean_score", metrics["mean_score"])
            artifact = Path(mlflow_dir) / "candidate.json"
            artifact.write_text(json.dumps(candidate), encoding="utf-8")
            mlflow.log_artifact(str(artifact))
            tracked_id = run.info.run_id
        tracked = MlflowClient().get_run(tracked_id)
        print({"tracked": True, "run_id": tracked.info.run_id, "metrics": tracked.data.metrics})
except Exception as exc:
    print({"tracked": False, "offline_fallback": True, "reason": type(exc).__name__})

## Record the generated shape and tutorial contract.


In [ ]:
active = next(
    (globals().get(name) for name in ("data", "baseline") if globals().get(name) is not None), None
)
assert active is not None
summary = {
    "tutorial_id": 18,
    "payments": len(active.behavior.payments),
    "events": len(active.behavior.payment_events),
}
print(summary)
assert summary["payments"] >= 0

In [ ]:
assert registry["Production"]["schema"] == "pit-v1"
print({"offline_fallback": True, "production": registry["Production"]["version"]})